# Stochastic Simulation — Detailed Summary

## Chapter 1 — Introduction

### 1.1 The Sampling Problem
- **Goal**: draw X^(i) ~ p*, i=1,…,N accurately
- Often only have unnormalised p̄*(x) with p*(x) = p̄*(x)/Z, where Z = ∫p̄*(x)dx is intractable
- Notation: (φ, p*) = E_{p*}[φ(X)] = ∫φ(x)p*(x)dx

### Applications
1. **Simulation of complex systems** (forward simulation)
2. **Bayesian statistical inference** (conditioning on data to find hidden variables)
3. **Generative modelling** (sampling from unknown distribution given dataset)

### Motivating Example: Estimating π
- Circle inscribed in square [−1,1]²: P(B) = π/4 where B = {(x,y): x²+y² ≤ 1}
- Monte Carlo: P(B) = E_P[1_B(X)] ≈ N_c/N → π/4
- Converges as N → ∞

## Chapter 1 — Bayesian Inference Primer

### 1.3.1 Bayes' Rule
**p(x|y) = p(y|x)p(x) / p(y)**

| Component | Name | Role |
|-----------|------|------|
| p(x) | Prior | Encodes prior knowledge about latent variable |
| p(y\|x) | Likelihood | Model of observation process |
| p(x\|y) | Posterior | Updated belief after data |
| p(y) | Marginal likelihood / Evidence | Normalising constant; model comparison |

**Remark 1.1**: Maps to sampling framework — target p*(x) := p(x|y), unnormalised p̄*(x) := p(y|x)p(x), normalising constant Z := p(y).

### Example 1.1 — Gaussian Conjugacy (1D)
Prior: p(x) = N(μ₀, σ₀²), Likelihood: p(y|x) = N(y; x, σ²)

**Posterior**: p(x|y) = N(μ_p, σ_p²) where:
- μ_p = (σ²μ₀ + σ₀²y) / (σ² + σ₀²)
- σ_p² = σ²σ₀² / (σ² + σ₀²)

Derivation: expand exp{−(y−x)²/(2σ²) − (x−μ₀)²/(2σ₀²)}, complete the square in x.

### Example 1.2 — Multivariate Gaussian Conjugacy
Prior: p(x) = N(μ₀, Σ₀), Likelihood: p(y|x) = N(Ax, Σ)

**Posterior**: x|y ~ N(μ_p, Σ_p) where:
- Σ_p = (A^TΣ⁻¹A + Σ₀⁻¹)⁻¹
- μ_p = Σ_p(A^TΣ⁻¹y + Σ₀⁻¹μ₀)

**Woodbury alternative**:
- Σ_p = Σ₀ − Σ₀A^T(AΣ₀A^T + Σ)⁻¹AΣ₀
- μ_p = μ₀ + Σ₀A^T(AΣ₀A^T + Σ)⁻¹(y − Aμ₀)

### Example 1.3 — Poisson-Gamma Conjugacy
Prior: Gamma(α, β), Likelihood: Pois(y; x)

**Posterior**: Gamma(α + y, β + 1)

### Proposition 1.1 — Conditional Bayes Rule
p(x|y,z) = p(y|x,z)p(x|z) / p(y|z)

### 1.3.2 Conditional Independence
**Definition 1.1**: X ⊥ Y | Z iff p(x,y|z) = p(x|z)p(y|z)

Bayes update with conditionally i.i.d. observations:
- p(x|y_{1:n}) ∝ p(x) ∏ᵢ p(yᵢ|x)

### 1.3.3 Marginal Likelihood
p(y) = ∫p(y|x)p(x)dx

**Example 1.4** (Gaussian): p(y) = N(y; μ₀, σ₀² + σ²)

**Example 1.5** (Model comparison): compare p₀(y) vs p₁(y) — model with higher marginal likelihood is preferred for given y.

## Chapter 2 — Generating Uniform Random Variates

### Pseudo-Random Numbers (Definition 2.1)
Deterministic sequence whose statistical properties match true random sequence from desired distribution. Advantages: repeatable, fast, cheap.

### Linear Congruential Generator (LCG)
- x_{n+1} ≡ ax_n + b (mod m)
- u_n = x_n/m ∈ [0,1)
- x₀ = seed, m = modulus, a = multiplier, b = shift
- b = 0: multiplicative; b ≠ 0: mixed generator
- Period T ≤ m; goal: T = m (full period)
- m ~ 2³² for single precision
- **Shortcomings**: periodic, fails in high dimensions (lattice structure in (u_k, u_{k+1}) plots)
- Modern standard: **Mersenne-Twister** (not an LCG)

## Chapter 2 — Inverse Transform Method

### Theorem 2.1 (Probability Integral Transform)
If X has CDF F_X, then F_X⁻¹(U) with U ~ Unif(0,1) has the same distribution as X.

**Proof**: P(F_X⁻¹(U) ≤ x) = P(U ≤ F_X(x)) = F_X(x). ∎

**Algorithm 1**: For i=1,…,n: generate U_i ~ Unif(0,1), set X_i = F_X⁻¹(U_i).

### Worked Examples

| Distribution | CDF F(x) | Inverse F⁻¹(u) | Sampler |
|-------------|----------|-----------------|--------|
| Discrete on {s₁,…,s_K} with weights w_k | F(s_k) = Σ_{j=1}^k w_j | Find min k: F(s_k) ≥ U | Staircase inversion |
| Exp(λ) | 1 − e^{−λx} | −(1/λ)ln(1−U) | U~Unif, X = −ln(1−U)/λ |
| Cauchy | ½ + π⁻¹arctan(x) | tan(π(U − ½)) | U~Unif, X = tan(π(U−½)) |
| Poisson(λ) | e^{−λ}Σ_{i=0}^k λⁱ/i! | Smallest k: F(k) ≥ U | Sequential search |
| Weibull(k) | 1 − e^{−x^k} | (−ln(1−U))^{1/k} | U~Unif, X = (−ln(1−U))^{1/k} |

**Limitation**: requires closed-form F⁻¹ (e.g. no closed form for Gaussian CDF).

## Chapter 2 — Transformation Method

### General Principle
Design g such that X = g(U) has target distribution. Need the transformation of random variables formula:

**1D**: p_Y(y) = p_X(g⁻¹(y)) |dg⁻¹(y)/dy|

**nD**: p_Y(y) = p_X(g⁻¹(y)) |det J_{g⁻¹}(y)|, where J_{g⁻¹} is the Jacobian of the inverse.

### Example 2.5 — Deriving the Box-Müller Transform
Start with X₁ ~ Exp(1/2), X₂ ~ Unif(−π,π). Transform:
- Y₁ = √X₁ cos X₂, Y₂ = √X₁ sin X₂
- Inverse: x₁ = y₁²+y₂², x₂ = arctan(y₂/y₁)
- Jacobian: |det J_{g⁻¹}| = 2
- Result: p_{Y₁,Y₂} = N(y₁;0,1)N(y₂;0,1) ✓

### Box-Müller Algorithm (Section 2.2.3)
Let U₁, U₂ ~ Unif(0,1) independent. Then:
- Z₁ = √(−2 ln U₁) cos(2πU₂)
- Z₂ = √(−2 ln U₁) sin(2πU₂)
are independent N(0,1). Standard method for Gaussian sampling in practice.

### Example 2.6 — Uniform on Disk
r ~ Unif(0,1), θ ~ Unif(−π,π):
- X₁ = √r cos θ, X₂ = √r sin θ → Uniform on unit disk (density = 1/π)
- **Caution**: X₁ = r cos θ (without √r) gives non-uniform distribution p = 1/(2π√(x₁²+x₂²))

### Example 2.7 — N(μ, σ²) from N(0,1)
g(x) = σx + μ. Verify via 1D transformation formula: p_Y(y) = N(y; μ, σ²). ✓

## Chapter 2 — Composition Methods

### 2.3.1 Sampling Joint Distributions
**Proposition 2.1 (Chain Rule for Sampling)**: To sample (X₁,X₂) ~ p(x₁,x₂):
1. X₁ ~ p_{X₁}(·)
2. X₂ | X₁=x₁ ~ p_{X₂|X₁}(·|x₁)

For n variables: sample sequentially X₁, then X₂|X₁, then X₃|X₁,X₂, etc.

**Example 2.8 (Linear Model)**: p(x)=Unif(−10,10), p(y|x)=N(ax+b, σ²) → generate (x,y) pairs with linear relationship + noise.

### 2.3.2 Sampling Marginals
**Proposition 2.2 (Marginalisation by Projection)**: Sample from p(x₁,x₂), keep only X₁ → samples from marginal p(x₁).

**Proof**: P(Z ∈ A) = ∫_{ℝ^{d₁}} ∫_A p(x₁,x₂) dx₁ dx₂ = ∫_A p_{X₂}(x₂) dx₂. ∎

### Discrete Mixture Sampling
For p*(x) = Σ_k w_k q_k(x):
1. k ~ Categorical(w₁,…,w_K) (using inversion)
2. X ~ q_k(x)

### 2.4 Multivariate Gaussian Sampling
To sample Y ~ N(μ, Σ) where Σ ∈ ℝ^{d×d}:
1. Cholesky: Σ = LL^T
2. v = [v₁,…,v_d]^T with each v_k ~ N(0,1) independently
3. Y = μ + Lv

Generalises 1D: Y = μ + σX where X ~ N(0,1).

## Chapter 2 — Rejection Sampling

### Theorem 2.2 (Fundamental Theorem of Simulation)
Sampling X ~ p*(x) is equivalent to sampling (X,Y) uniformly on the region A = {(x,y): 0 ≤ y ≤ p̄*(x)} and keeping the x-marginal.

**Proof**: Joint q(x,y) = (1/Z)1_A(x,y). Marginal: q(x) = ∫₀^{p̄*(x)} (1/Z)dy = p̄*(x)/Z = p*(x). ∎

Works for unnormalised densities p̄* too.

### Rejection Sampling Algorithm
Choose proposal q(x) and constant M such that p̄*(x) ≤ Mq(x) for all x.

**Algorithm 5** (normalised) / **Algorithm 6** (unnormalised):
1. Sample X' ~ q(x)
2. Sample U ~ Unif(0,1)
3. If U ≤ p̄*(X')/(Mq(X')): **accept** X'; else: **reject**

### Proposition 2.3 (Acceptance Rate)
- Normalised target: **â = 1/M** (where M ≥ 1)
- Unnormalised target: **â = Z/M** (where Z = ∫p̄*dx)

**Proof**: â = E[a(X')] = ∫(p̄*(x)/(Mq(x)))q(x)dx = Z/M. For normalised: Z=1. ∎

### Optimal M and Proposal Design
- **Optimal M**: M* = sup_x p*(x)/q(x) (smallest covering constant)
  - Find x* = argmax p*/q (take log, differentiate, set to zero)
  - Then M* = p*(x*)/q(x*)
- **Optimal proposal**: parameterise q_θ, then θ* = argmin_θ log M_θ

### Key Examples

| Example | Target | Proposal | Key Result |
|---------|--------|----------|------------|
| 2.10 | Beta(2,2) | Box [0,1]×[0,1.5] | â = 1/1.5 = 0.67 |
| 2.11 | Beta(2,2) | N(0.5, 0.25) | M=1.3, â = 0.77 (better) |
| 2.12 | Truncated N(0,1) on [−a,a] | N(0,1) | M=1, â = Z = ∫_{−a}^a N(y;0,1)dy |
| 2.13 | Gaussian posterior | N(0,1) | Rejection from prior without computing posterior |
| 2.14 | exp(−x²/2) | Cauchy | M = 2e^{−1/2}, x*=±1 |
| 2.15 | Gamma(α,1) | Exp(λ) | Optimal λ*=1/α, M*=e^{−(α−1)}/Γ(α) |
| 2.16 | Poisson-Gamma posterior | Exp(λ) | Optimal λ*=2/(α+y) |

## Chapter 2 — Curse of Dimensionality

### Why Rejection Fails in High Dimensions
For unit ball inside unit cube:
- â_d = Vol(ball) / Vol(cube) = π^{d/2} / Γ(d/2+1)
- â₂ ≈ 0.78, â₃ ≈ 0.52, and decays **super-exponentially** in d

### General Result
If p*(x) = ∏p₀(xᵢ) and q(x) = ∏q₀(xᵢ) with K = sup p₀/q₀, then M = K^d → acceptance rate â = 1/K^d → 0 as d → ∞.

### Practical Issues
- Product ∏p(yᵢ|X') underflows for moderate n → use **log-space**: log a = log p̄*(X') − log M − log q(X'), accept when log U ≤ log a
- Global bound M may not exist for complex posteriors

### Mitigation
- If product structure exists: sample coordinate-by-coordinate → cost O(dK) instead of O(K^d)
- Motivates advanced methods: MCMC, SMC (covered in later chapters)